# Regional Ocean Grids Overview

Creates ROMS grids for Eastern Boundary Upwelling Systems (EBUS), Gulf of Guinea,
Arabian Sea, U.S. East Coast, Kuroshio, and the Mediterranean, then plots all
domains on a global Robinson projection.

In [ ]:
import yaml
import cartopy
import cartopy.crs as ccrs
import cartopy.feature
import cmocean
import matplotlib.pyplot as plt
import roms_tools as rt

## Load grid configurations and build grids

Reads `grids.yaml` (list of dicts) and constructs an `rt.Grid` for each entry.
Grid creation interpolates ETOPO5 bathymetry — expect ~10–30 s per grid.

In [ ]:
def read_yaml(filepath):
    with open(filepath) as f:
        return yaml.safe_load(f)


grid_kwargs_list = read_yaml("grids.yaml")

grids = {}
for kw in grid_kwargs_list:
    name = kw.pop("name")
    print(f"Building grid: {name} ...", end=" ", flush=True)
    grids[name] = rt.Grid(**kw)
    print("done")

print(f"\nCreated {len(grids)} grids: {list(grids.keys())}")

## Global plot of all grid domains

In [ ]:
prj = ccrs.Robinson(central_longitude=305.0)
fig, ax = plt.subplots(
    figsize=(16, 9), facecolor="k", subplot_kw=dict(projection=prj)
)

ax.add_feature(
    cartopy.feature.NaturalEarthFeature(
        "physical", "ocean", "110m",
        edgecolor="face", facecolor="#749EB0",
    ),
    zorder=-9999,
)
ax.add_feature(
    cartopy.feature.NaturalEarthFeature(
        "physical", "land", "110m",
        edgecolor="face", facecolor="#355560",
    )
)

ax.coastlines(linewidth=0.5, color="white")
ax.spines["geo"].set_visible(True)
ax.spines["geo"].set_linewidth(2)
ax.spines["geo"].set_edgecolor("black")

for name, g in grids.items():
    ax.pcolormesh(
        g.ds.lon_rho,
        g.ds.lat_rho,
        g.ds.h,
        transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.deep,
        vmin=0,
        vmax=5000,
        shading="auto",
    )

ax.set_global()
fig.tight_layout()
fig.savefig("assets/cson.png", dpi=300, transparent=True)
plt.show()
print("Saved to assets/cson.png")